# Swlc SAM archive — Colab fix for iPhone (read-only)

This notebook replaces the earlier V2 notebook that failed at `assert candidate.is_file()` because Colab couldn't locate a Python script in mounted Google Drive.

**What changed:** Required Python source code is downloaded directly from a *fixed, immutable GitHub commit* and **SHA-256 verified before import**. Google Drive is still used for archive output only. This avoids depending on Colab's account-dependent `MyDrive` source-folder layout. The dependency cell installs only missing Python packages instead of pinning cryptography to a version that may conflict with Colab's existing `pyOpenSSL`.

**Security:** Inspect these code cells before running. Google Drive access grants this runtime permission to read/write mounted Drive data. No passwords or PostgreSQL credentials are in the notebook. The final step asks for the original Swlc PostgreSQL connection URI and a separately stored 20+ character archive passphrase through masked prompts.

**DO NOT run the previous notebook's failing cells. Start with a new runtime here.**

**No deletion, truncation, VACUUM, Supabase database migration, billing change, or scheduler activation is performed.** Even a completed archive requires independent downloaded-byte verification and an isolated restore before a separate retention decision.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted. Source scripts will come from verified GitHub, not from this mount.")

In [ ]:
# Avoid Colab dependency conflicts: install only libraries not already present.
# Do not override the runtime's cryptography version unless it is missing.
import importlib.util
import subprocess
import sys
needed = []
if importlib.util.find_spec("psycopg") is None:
    needed.append("psycopg[binary]==3.2.10")
if importlib.util.find_spec("cryptography") is None:
    needed.append("cryptography")
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *needed])
import psycopg
import cryptography
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
print("Dependencies available:", "psycopg", psycopg.__version__, "cryptography", cryptography.__version__)

In [ ]:
# Download public source-only files from an immutable commit, hash them, THEN import.
# Stop on any fetch/hash mismatch. Never download secrets, record rows or credentials.
import hashlib
from pathlib import Path
import sys
from urllib.request import urlopen

REPOSITORY_COMMIT = "9e9ce8982b810ce6a4732ddf4220dc37c8dc190e"
SOURCE_ROOT = (
    "https://raw.githubusercontent.com/bookieandco/crispy-waddle/"
    + REPOSITORY_COMMIT + "/scripts/swlc-drive-archive/"
)
SOURCE_HASHES = {
    "swlc_sam_drive_archive.py": "090958b0c31e34fbfd28bf7d6722ec074ac34c947859591256647d696785a0c9",
    "swlc_sam_archive_verify.py": "2bde7cc34f91fa184efae7b6895fae8b40600f9dc389c2fb4a60c20003050123",
}
target = Path("/content/swlc_sam_verified_code")
if not target.exists():
    target.mkdir(mode=0o700, parents=False)
for filename, expected_hash in SOURCE_HASHES.items():
    with urlopen(SOURCE_ROOT + filename, timeout=30) as response:
        source = response.read(100_001)
    if len(source) > 100_000 or hashlib.sha256(source).hexdigest() != expected_hash:
        raise RuntimeError("GitHub code hash mismatch. STOP: " + filename)
    out = target / filename
    if out.exists() and hashlib.sha256(out.read_bytes()).hexdigest() != expected_hash:
        raise RuntimeError("Unexpected existing local source. Restart runtime before continuing.")
    if not out.exists():
        out.write_bytes(source)
if str(target) not in sys.path:
    sys.path.insert(0, str(target))
from swlc_sam_drive_archive import colab_operator_main, DRIVE_FOLDER, PROJECT_REF
print("Verified exact reviewed source for project:", PROJECT_REF)
print("Expected encrypted archive destination:", DRIVE_FOLDER)

In [ ]:
# Check the exact archive folder via Google's API, independent of mounted paths.
# If access fails, reauthenticate with the Google account that owns JHADINA-HOMEBASE.
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
creds, _ = google.auth.default()
api = build("drive", "v3", credentials=creds, cache_discovery=False)
try:
    archive = api.files().get(
        fileId="1UE-kLmY83gK_rt-jtHGgA3VQFj89jdNE",
        fields="id,name,mimeType,parents,trashed,capabilities(canAddChildren)",
        supportsAllDrives=True,
    ).execute()
except Exception as exc:
    raise RuntimeError(
        "Colab's Google account cannot access the JHADINA-HOMEBASE archive folder. "
        "Sign in to the account that opens the folder link, restart this Colab runtime, "
        "and run the notebook again. No database connection has been attempted."
    ) from exc
assert (
    archive.get("name") == "02-ENCRYPTED-ARCHIVES"
    and not archive.get("trashed")
    and "1b-DZZOwzzavYS-7uh8qIi7vgZeR9QnyD" in archive.get("parents", [])
), "Unexpected Google Drive folder identity: stop before export."
print("SUCCESS: Colab's Google API account can see the verified destination folder.")

In [ ]:
# Confirm the Google account and Drive archive destination BEFORE entering any credentials.
from pathlib import Path
from swlc_sam_drive_archive import DRIVE_FOLDER
root = Path("/content/drive/MyDrive")
if not root.is_dir():
    raise RuntimeError(
        "Colab does not show /content/drive/MyDrive. Re-run the Drive mount cell, "
        "check your Google account, or open the notebook on your MacBook."
    )
if not DRIVE_FOLDER.is_dir():
    raise RuntimeError(
        "Colab can access the verified destination through the Drive API, but its mounted "
        "filesystem does not show that folder at /content/drive/MyDrive. "
        "Disconnect the Colab runtime, reconnect and mount the same Google account, "
        "then rerun from the top. Do not export into a newly invented folder."
    )
print("Archive destination exists. No credentials collected yet.")
print("Ready for owner-run, read-only export.")

In [ ]:
# LAST CELL: Run once, only after the above checks PASS.
# Input prompts are masked. Don't paste credentials into a notebook cell, chat, GitHub or Drive.
colab_operator_main()

### After the export

The first genuine exported archive should create two files in your private `JHADINA-HOMEBASE/04-SUBSYSTEM-ARCHIVES/SWLC-SAM-ARCHIVE-STAGING-2026-10-12/02-ENCRYPTED-ARCHIVES` folder:

- `swlc-sam-versions-before-...csv.gz.aesgcm` (encrypted row data)
- The same filename suffixed `.json` (manifest)

If those files are not present, the export is **not complete**. The archive's internal validation is not equivalent to independent Drive readback or PostgreSQL restoration. Never delete any Supabase data based solely on this notebook.

If an error remains, share **only the error message**, with all passwords, tokens and database connection strings redacted.